# H7. Чувствительность к классу модели и инициализации

## Постановка

Диагностика в [REPORT.md §7.1(a)](diagnostic/REPORT.md) показала, что при
**случайно инициализированном** MLP-классификаторе `RecModel` поверхность
скорa $\hat r(u, v)$ практически не зависит от $u$: top-K выдача оказывается
общей для всех пользователей, что *унифицирует* динамику режимов
`closed_loop`/`static`/`fresh_oracle`. Гипотеза:

> **H7.** Если рекомендатель *с самого начала* различает пользователей —
> либо за счёт более «прозрачной» геометрии скорa (bilinear matrix
> factorization), либо благодаря **прогретой** инициализации на истинных
> предпочтениях, — разрыв между петлевым (`closed_loop`) и непетлевыми
> режимами по метрикам $\mathrm{tr}(\hat\Sigma)$ и $KL(P_T\|P_0)$
> увеличивается.

В эксперименте варьируется

- **класс модели:** `MLP` (`RecModel`, 64 скрытых нейрона, ≈600 параметров)
  vs **`MF`** (`MFModel`, $\sigma(u^\top M v + b)$, $M\in\mathbb R^{d\times d}$,
  $d^2+1 = 65$ параметров);
- **инициализация:**
  - `cold` — случайные веса (как в H1/H3);
  - `warm` — pretrain N_PRE эпох на ground-truth матрице `true_pref`;
  - `warm_noisy` — то же, но на `true_pref + N(0, σ_noise²)` (имитирует
    исторические клики до старта симуляции);
- **режим симуляции:** все четыре, как в H1/H3;
- **seed:** 30 случайных запусков для оценки доверительных интервалов.

Полный grid: $2 \times 3 \times 4 \times 30 = 720$ запусков по $T=100$ шагов.


In [1]:
import sys, os, time
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from itertools import product

from sim.user_generator import GMMUserGenerator
from sim.environment    import SimulationEnvironment, ExperimentDataset
from sim.click_model    import ClickModel
from models.rec_models  import RecModel, MFModel
from models.serving     import ServingPolicy

FIGURES_DIR = Path('../../paper/figures')
RESULTS_DIR = Path('results')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK')

Imports OK


In [2]:
# ── Конфигурация ─────────────────────────────────────────────────────
SMOKE = False     # True → быстрый прогон (1 seed) для проверки пайплайна

EMB_DIM     = 8
N_USERS     = 300
N_ITEMS     = 300
K_REC       = 10
T           = 100
T_RET       = 10
REPLACE     = 0.05
ADHERENCE   = 0.7
USER_DRIFT  = 0.005    # β
DRIFT_ALPHA = 0.02
INTER_DIST  = 5.0
SIGMA_K     = 0.8
K_GMM       = 3

# Warm-start
N_PRETRAIN_EPOCHS = 25
PRETRAIN_LR       = 3e-3
PRETRAIN_SUBSAMPLE = 6000   # пары (u, i) для предобучения
WARM_NOISE_STD    = 0.10

N_SEEDS = 1 if SMOKE else 30

MODES        = ['closed_loop', 'static', 'fresh_oracle', 'no_influence']
MODEL_TYPES  = ['MLP', 'MF']
INIT_TYPES   = ['cold', 'warm', 'warm_noisy']

COLORS = {'closed_loop': '#d62728', 'static': '#ff7f0e',
          'fresh_oracle': '#2ca02c', 'no_influence': '#1f77b4'}
LABELS = {'closed_loop': r'closed\_loop',
          'static':       'static',
          'fresh_oracle': r'fresh\_oracle',
          'no_influence': r'no\_influence ($\alpha=0$)'}

PARAMS = dict(
    N_USERS=N_USERS, N_ITEMS=N_ITEMS, EMB_DIM=EMB_DIM,
    K_GMM=K_GMM, K_REC=K_REC, T=T, T_RET=T_RET,
    REPLACE=REPLACE, ADHERENCE=ADHERENCE, USER_DRIFT=USER_DRIFT,
    DRIFT_ALPHA=DRIFT_ALPHA, INTER_DIST=INTER_DIST, SIGMA_K=SIGMA_K)

print(f'N={N_USERS}, items={N_ITEMS}, d={EMB_DIM}, T={T}, '
      f'α={ADHERENCE}, β={USER_DRIFT}, α_drift={DRIFT_ALPHA}')
print(f'Grid: {len(MODEL_TYPES)} models × {len(INIT_TYPES)} inits × '
      f'{len(MODES)} modes × {N_SEEDS} seeds = '
      f'{len(MODEL_TYPES)*len(INIT_TYPES)*len(MODES)*N_SEEDS} runs')

N=300, items=300, d=8, T=100, α=0.7, β=0.005, α_drift=0.02
Grid: 2 models × 3 inits × 4 modes × 30 seeds = 720 runs


## Параметры и допущения

| Параметр | Значение | Смысл |
|----------|---------|-------|
| `EMB_DIM` | 8 | Размерность пространства эмбеддингов; для `MFModel` это $d$ в $M\in\mathbb R^{d\times d}$ |
| `N_USERS`, `N_ITEMS` | 300, 300 | Размер активной популяции и каталога |
| `T`, `T_RET` | 100, 10 | Длина симуляции и период переобучения в `closed_loop` |
| `ADHERENCE` ($\alpha$) | 0.7 | Сила петли обратной связи в `ClickModel` |
| `USER_DRIFT` ($\beta$) | 0.005 | β-дрейф эмбеддингов к рекомендованным айтемам |
| `DRIFT_ALPHA` | 0.02 | Дрейф GMM-центров (активен только в `closed_loop`) |
| `N_PRETRAIN_EPOCHS` | 25 | Эпохи warm-start обучения на `true_pref` |
| `WARM_NOISE_STD` | 0.10 | СКО гауссовского шума, добавляемого к `true_pref` в `warm_noisy` |
| `N_SEEDS` | 30 | Бутстрап-выборка случайных запусков |

Замечание про `MFModel`: при `identity_init=True` (используется во всех
конфигурациях) начальная функция скорa $\hat r(u, v) = \sigma(u^\top v)$
**сразу** различает пользователей геометрически — без обучения. Это меняет
качество стартовых рекомендаций по сравнению с MLP, у которого случайные
веса разрушают зависимость от $u$.


In [3]:
# ── Утилиты: подготовка данных и моделей ────────────────────────────
def make_gmm_params(K, dim, inter_dist, sigma):
    means, covs, weights = [], [], []
    for k in range(K):
        angle = 2 * np.pi * k / K
        m = np.zeros(dim)
        m[0] = inter_dist * np.cos(angle)
        m[1] = inter_dist * np.sin(angle)
        means.append(m)
        covs.append(sigma**2 * np.eye(dim))
        weights.append(1.0 / K)
    return means, covs, weights


def make_items(N_items, K, means, sigma, dim, rng):
    parts = []
    for k in range(K):
        n_k = N_items // K if k < K - 1 else N_items - (N_items // K) * (K - 1)
        parts.append(rng.multivariate_normal(means[k], sigma**2 * np.eye(dim), n_k))
    return np.vstack(parts)


def make_true_pref(users, items, tau=2.0):
    scores = users @ items.T / tau
    return 1 / (1 + np.exp(-scores))


def make_model(model_type, dim):
    """Возвращает свежую модель нужного типа."""
    if model_type == 'MLP':
        return RecModel(dim, dim, hidden_size=64)
    if model_type == 'MF':
        return MFModel(dim, dim, identity_init=True)
    raise ValueError(model_type)


class _PairBCEDataset(Dataset):
    """Пары (u, i, target) для предобучения; target ∈ [0,1] непрерывный."""
    def __init__(self, users_emb, items_emb, target_matrix, subsample, rng):
        n_u, n_i = target_matrix.shape
        all_pairs = np.array([(u, i) for u in range(n_u) for i in range(n_i)])
        if subsample is not None and subsample < len(all_pairs):
            idx = rng.choice(len(all_pairs), subsample, replace=False)
            all_pairs = all_pairs[idx]
        self.pairs    = all_pairs.astype(np.int64)
        self.targets  = target_matrix[self.pairs[:, 0], self.pairs[:, 1]].astype(np.float32)
        self.users_emb = users_emb.astype(np.float32)
        self.items_emb = items_emb.astype(np.float32)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u, i = self.pairs[idx]
        return (torch.from_numpy(self.users_emb[u]),
                torch.from_numpy(self.items_emb[i]),
                torch.tensor(self.targets[idx]))


def pretrain_model(model, user_emb, item_emb, target_matrix,
                   n_epochs, lr, subsample, rng, device):
    """Обучает модель на матрице целей `target_matrix` (значения в [0,1]).

    Используется для warm-start: target_matrix = true_pref (для warm) или
    clip(true_pref + N(0, σ²), 0, 1) (для warm_noisy).
    """
    ds = _PairBCEDataset(user_emb, item_emb, target_matrix, subsample, rng)
    loader = DataLoader(ds, batch_size=512, shuffle=True)
    opt = optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCELoss()
    model.train()
    losses = []
    for _ in range(n_epochs):
        ep_loss, n_batches = 0.0, 0
        for u_b, i_b, t_b in loader:
            u_b, i_b, t_b = u_b.to(device), i_b.to(device), t_b.to(device)
            opt.zero_grad()
            pred = model(u_b, i_b)
            loss = crit(pred, t_b)
            loss.backward()
            opt.step()
            ep_loss += loss.item(); n_batches += 1
        losses.append(ep_loss / max(n_batches, 1))
    return losses


def build_env(mode, seed, params, model_type, init_type):
    """Полный билд окружения с заданным классом модели и инициализацией."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed + 1000)

    means, covs, weights = make_gmm_params(
        params['K_GMM'], params['EMB_DIM'],
        params['INTER_DIST'], params['SIGMA_K'])
    gen = GMMUserGenerator(
        component_means=means, component_covs=covs, component_weights=weights,
        replacement_rate=params['REPLACE'], memory_effect=6)

    np.random.seed(seed)
    user_emb, _ = gen.initialize(params['N_USERS'])
    item_emb = make_items(params['N_ITEMS'], params['K_GMM'], means,
                          params['SIGMA_K'], params['EMB_DIM'], rng)
    true_pref = make_true_pref(user_emb, item_emb)
    matrix    = np.full((params['N_USERS'], params['N_ITEMS']), np.nan)
    dataset   = ExperimentDataset(user_emb.copy(), item_emb.copy(), matrix)

    device = torch.device('cpu')
    model  = make_model(model_type, params['EMB_DIM']).to(device)

    if init_type == 'cold':
        pass
    elif init_type == 'warm':
        pretrain_model(model, user_emb, item_emb, true_pref,
                       n_epochs=N_PRETRAIN_EPOCHS, lr=PRETRAIN_LR,
                       subsample=PRETRAIN_SUBSAMPLE, rng=rng, device=device)
    elif init_type == 'warm_noisy':
        noisy = np.clip(true_pref + rng.normal(0, WARM_NOISE_STD, true_pref.shape), 0.0, 1.0)
        pretrain_model(model, user_emb, item_emb, noisy,
                       n_epochs=N_PRETRAIN_EPOCHS, lr=PRETRAIN_LR,
                       subsample=PRETRAIN_SUBSAMPLE, rng=rng, device=device)
    else:
        raise ValueError(init_type)

    policy = ServingPolicy('top_k')
    alpha_c = 0.0 if mode == 'no_influence' else params['ADHERENCE']
    click   = ClickModel(adherence=alpha_c, usage_rate=0.8, noise_level=0.05)
    d_alpha = params['DRIFT_ALPHA'] if mode == 'closed_loop' else 0.0

    env = SimulationEnvironment(
        dataset=dataset, rec_model=model,
        user_generator=gen, click_model=click,
        serving_policy=policy, mode=mode,
        true_preference_matrix=true_pref,
        retrain_period=params['T_RET'], K=params['K_REC'],
        device=device, seen_filter=True,
        user_drift_beta=params['USER_DRIFT'],
        drift_alpha=d_alpha)
    return env

print('Utility functions ready')

Utility functions ready


In [4]:
# ── Главный прогон: 2 models × 3 inits × 4 modes × N_SEEDS ───────────
# Результат: dict[(model_type, init_type, mode)] -> list[DataFrame]
results = {}
combos = list(product(MODEL_TYPES, INIT_TYPES, MODES))
total_runs = len(combos) * N_SEEDS

t_start = time.time()
run_idx = 0
for model_type, init_type, mode in combos:
    key = (model_type, init_type, mode)
    results[key] = []
    for seed in range(N_SEEDS):
        run_idx += 1
        t0 = time.time()
        env = build_env(mode, seed, PARAMS, model_type, init_type)
        for t in range(T):
            env.step(t)
        df = env.metrics.get_dataframe()
        df['model_type'] = model_type
        df['init_type']  = init_type
        df['mode']       = mode
        df['seed']       = seed
        results[key].append(df)
        elapsed = time.time() - t0
        total_elapsed = time.time() - t_start
        eta = total_elapsed / run_idx * (total_runs - run_idx)
        if seed == 0 or seed == N_SEEDS - 1 or seed % 10 == 0:
            print(f'  [{run_idx:3d}/{total_runs}] {model_type:3s} {init_type:10s} '
                  f'{mode:14s} seed={seed:2d} '
                  f'trΣ:{df.trace_sigma.iloc[0]:5.1f}→{df.trace_sigma.iloc[-1]:6.2f} '
                  f'KL:{df.kl_from_initial.iloc[-1]:5.2f}  '
                  f'({elapsed:.1f}s, ETA {eta/60:.1f}m)')
print(f'\nAll runs done in {(time.time()-t_start)/60:.1f} min')

  [  1/720] MLP cold       closed_loop    seed= 0 trΣ: 27.9→  0.71 KL:17.38  (10.3s, ETA 123.7m)
  [ 11/720] MLP cold       closed_loop    seed=10 trΣ: 28.2→  0.66 KL:18.03  (9.5s, ETA 114.1m)
  [ 21/720] MLP cold       closed_loop    seed=20 trΣ: 27.7→  0.51 KL:17.96  (9.4s, ETA 111.7m)
  [ 30/720] MLP cold       closed_loop    seed=29 trΣ: 28.2→  0.98 KL:16.98  (9.6s, ETA 109.9m)
  [ 31/720] MLP cold       static         seed= 0 trΣ: 27.9→  0.64 KL:17.80  (8.2s, ETA 109.2m)
  [ 41/720] MLP cold       static         seed=10 trΣ: 28.2→  0.53 KL:17.48  (8.3s, ETA 104.5m)
  [ 51/720] MLP cold       static         seed=20 trΣ: 27.7→  0.87 KL:16.96  (8.2s, ETA 101.0m)
  [ 60/720] MLP cold       static         seed=29 trΣ: 28.2→  0.43 KL:17.84  (8.2s, ETA 98.4m)
  [ 61/720] MLP cold       fresh_oracle   seed= 0 trΣ: 27.9→  1.00 KL:16.60  (9.0s, ETA 98.3m)
  [ 71/720] MLP cold       fresh_oracle   seed=10 trΣ: 28.2→  0.55 KL:17.86  (9.1s, ETA 96.6m)
  [ 81/720] MLP cold       fresh_oracle   

In [5]:
# ── Агрегация в длинный DataFrame ────────────────────────────────────
all_df = pd.concat([df for k in results for df in results[k]], ignore_index=True)
print('Combined shape:', all_df.shape)
all_df.head(3)

Combined shape: (72000, 23)


,t,intra_list_diversity,catalog_coverage,exposure_entropy,gini_exposure,trace_sigma,log_det_sigma,leading_eigenvalue,min_eigenvalue,intra_cluster_variance,...,kl_from_initial,kl_step,observed_quality,true_quality,bias_gap,exposure_dependence,model_type,init_type,mode,seed
0,0,2.982602,0.276667,3.838138,0.885713,27.888108,1.348207,12.556495,0.445175,0.559701,...,0.020344,0.000000,0.809333,0.475796,-0.333537,0.0,MLP,cold,closed_loop,0
1,1,2.969701,0.400000,4.329752,0.811824,26.220387,0.841585,11.797809,0.401135,0.526465,...,0.063165,0.020926,0.689000,0.447152,-0.241848,0.0,MLP,cold,closed_loop,0
2,2,3.187223,0.456667,4.497109,0.775336,24.793471,0.361739,11.554456,0.365160,0.497979,...,0.102642,0.019850,0.639000,0.436398,-0.202602,0.0,MLP,cold,closed_loop,0


In [6]:
# ── Сводка по последнему шагу ────────────────────────────────────────
last = all_df[all_df['t'] == all_df['t'].max()].copy()
summary = (last
    .groupby(['model_type', 'init_type', 'mode'])
    .agg(trace_T_mean=('trace_sigma', 'mean'),
         trace_T_std =('trace_sigma', 'std'),
         kl_T_mean   =('kl_from_initial', 'mean'),
         kl_T_std    =('kl_from_initial', 'std'),
         gini_T_mean =('gini_exposure', 'mean'),
         coverage_T  =('catalog_coverage', 'mean'),
         n_seeds     =('seed', 'nunique'))
    .reset_index())
summary['trace_drop_pct'] = (1 - summary['trace_T_mean'] /
                              all_df.groupby(['model_type', 'init_type', 'mode'])['trace_sigma']
                                    .first().reset_index(drop=True).values) * 100
summary_path = RESULTS_DIR / 'H7_mf_warmstart_summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved', summary_path)
summary.round(3)

Saved results/H7_mf_warmstart_summary.csv


,model_type,init_type,mode,trace_T_mean,trace_T_std,kl_T_mean,kl_T_std,gini_T_mean,coverage_T,n_seeds,trace_drop_pct
0,MF,cold,closed_loop,1.548,0.119,15.881,0.215,0.240,0.999,30,94.784
1,MF,cold,fresh_oracle,1.553,0.085,16.084,0.198,0.240,0.998,30,94.767
2,MF,cold,no_influence,8.932,0.778,7.663,0.604,0.317,0.999,30,69.902
3,MF,cold,static,1.512,0.104,15.983,0.228,0.240,0.999,30,94.905
4,MF,warm,closed_loop,1.504,0.125,15.938,0.246,0.252,0.999,30,94.998
5,MF,warm,fresh_oracle,1.550,0.110,16.094,0.221,0.234,0.999,30,94.846
6,MF,warm,no_influence,8.152,0.885,7.805,0.550,0.364,0.998,30,72.895
7,MF,warm,static,1.556,0.124,16.124,0.224,0.239,0.998,30,94.827
8,MF,warm_noisy,closed_loop,1.542,0.132,16.131,0.235,0.289,0.994,30,94.889
9,MF,warm_noisy,fresh_oracle,1.439,0.095,15.932,0.208,0.251,0.998,30,95.232


In [7]:
# ── Главный рисунок: tr(Σ) во времени для 6 конфигов модели×инициализации
fig, axes = plt.subplots(len(MODEL_TYPES), len(INIT_TYPES),
                         figsize=(13, 7), sharex=True, sharey=True)
for r, model_type in enumerate(MODEL_TYPES):
    for c, init_type in enumerate(INIT_TYPES):
        ax = axes[r, c]
        for mode in MODES:
            key = (model_type, init_type, mode)
            arr = np.stack([df['trace_sigma'].values for df in results[key]])
            ts  = results[key][0]['t'].values
            mu, sd = arr.mean(0), arr.std(0)
            ax.plot(ts, mu, color=COLORS[mode], lw=1.8, label=LABELS[mode])
            ax.fill_between(ts, mu - sd, mu + sd, color=COLORS[mode], alpha=0.15)
        ax.set_title(f'{model_type} · {init_type}', fontsize=11)
        ax.grid(True, ls='--', alpha=0.4)
        if r == len(MODEL_TYPES) - 1:
            ax.set_xlabel('Step $t$')
        if c == 0:
            ax.set_ylabel(r'$\mathrm{tr}(\hat{\Sigma}_t^u)$')
        if r == 0 and c == 0:
            ax.legend(fontsize=8, loc='upper right')
fig.suptitle(
    f'H7: tr(Σ̂) при разных моделях и инициализациях '
    f'(N={N_USERS}, T={T}, seeds={N_SEEDS})', y=1.00)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'H7_trace_sigma_grid.pdf', bbox_inches='tight')
plt.show()
print('Saved H7_trace_sigma_grid.pdf')

Saved H7_trace_sigma_grid.pdf


In [8]:
# ── Аналогичный рисунок для KL ───────────────────────────────────────
fig, axes = plt.subplots(len(MODEL_TYPES), len(INIT_TYPES),
                         figsize=(13, 7), sharex=True, sharey=True)
for r, model_type in enumerate(MODEL_TYPES):
    for c, init_type in enumerate(INIT_TYPES):
        ax = axes[r, c]
        for mode in MODES:
            key = (model_type, init_type, mode)
            arr = np.stack([df['kl_from_initial'].values for df in results[key]])
            ts  = results[key][0]['t'].values
            mu, sd = arr.mean(0), arr.std(0)
            ax.plot(ts, mu, color=COLORS[mode], lw=1.8, label=LABELS[mode])
            ax.fill_between(ts, mu - sd, mu + sd, color=COLORS[mode], alpha=0.15)
        ax.set_title(f'{model_type} · {init_type}', fontsize=11)
        ax.grid(True, ls='--', alpha=0.4)
        if r == len(MODEL_TYPES) - 1:
            ax.set_xlabel('Step $t$')
        if c == 0:
            ax.set_ylabel(r'$KL(P_t \| P_0)$')
        if r == 0 and c == 0:
            ax.legend(fontsize=8, loc='upper left')
fig.suptitle(
    f'H7: KL-дивергенция при разных моделях и инициализациях '
    f'(N={N_USERS}, T={T}, seeds={N_SEEDS})', y=1.00)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'H7_kl_grid.pdf', bbox_inches='tight')
plt.show()
print('Saved H7_kl_grid.pdf')

Saved H7_kl_grid.pdf


In [9]:
# ── Главный диагностический график: gap (closed_loop − static)
# по trace_T и KL_T для каждой пары (model, init)
import matplotlib.patches as mpatches

def gap_bootstrap(series_cl, series_st, n_boot=2000, q=(0.025, 0.975), rng=None):
    rng = rng or np.random.default_rng(0)
    cl = np.asarray(series_cl); st = np.asarray(series_st)
    idx = rng.integers(0, len(cl), size=(n_boot, len(cl)))
    diffs = cl[idx].mean(1) - st[idx].mean(1)
    return diffs.mean(), np.quantile(diffs, q[0]), np.quantile(diffs, q[1])

records = []
rng = np.random.default_rng(42)
for model_type in MODEL_TYPES:
    for init_type in INIT_TYPES:
        last_cl = [df['trace_sigma'].iloc[-1] for df in results[(model_type, init_type, 'closed_loop')]]
        last_st = [df['trace_sigma'].iloc[-1] for df in results[(model_type, init_type, 'static')]]
        kl_cl   = [df['kl_from_initial'].iloc[-1] for df in results[(model_type, init_type, 'closed_loop')]]
        kl_st   = [df['kl_from_initial'].iloc[-1] for df in results[(model_type, init_type, 'static')]]
        tr_gap, tr_lo, tr_hi = gap_bootstrap(last_cl, last_st, rng=rng)
        kl_gap, kl_lo, kl_hi = gap_bootstrap(kl_cl, kl_st, rng=rng)
        records.append(dict(
            model=model_type, init=init_type,
            trace_gap=tr_gap, trace_lo=tr_lo, trace_hi=tr_hi,
            kl_gap=kl_gap, kl_lo=kl_lo, kl_hi=kl_hi))
gap_df = pd.DataFrame(records)
print('closed_loop − static (95% bootstrap CI):')
print(gap_df.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
xpos = np.arange(len(gap_df))
labels = [f'{r.model}\n{r.init}' for r in gap_df.itertuples()]
for ax, metric, title in [
    (axes[0], 'trace', r'gap $\mathrm{tr}\hat\Sigma_T$:  closed\_loop $-$ static'),
    (axes[1], 'kl',    r'gap $KL_T$:  closed\_loop $-$ static')]:
    vals = gap_df[f'{metric}_gap'].values
    los  = np.maximum(vals - gap_df[f'{metric}_lo'].values, 0.0)
    his  = np.maximum(gap_df[f'{metric}_hi'].values - vals, 0.0)
    ax.bar(xpos, vals, yerr=[los, his], color='#4c72b0', capsize=4, alpha=0.85)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(xpos)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.grid(True, axis='y', ls='--', alpha=0.4)
fig.suptitle('H7: эффект режима как функция (модель, инициализация)', y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'H7_mode_gap.pdf', bbox_inches='tight')
plt.show()
print('Saved H7_mode_gap.pdf')
gap_df.to_csv(RESULTS_DIR / 'H7_mode_gap.csv', index=False)

closed_loop − static (95% bootstrap CI):
model       init  trace_gap  trace_lo  trace_hi  kl_gap  kl_lo  kl_hi
  MLP       cold      0.139     0.011     0.269  -0.043 -0.287  0.199
  MLP       warm     -0.162    -0.219    -0.109   0.793  0.641  0.945
  MLP warm_noisy     -0.234    -0.282    -0.181   0.731  0.630  0.835
   MF       cold      0.036    -0.014     0.087  -0.104 -0.193 -0.008
   MF       warm     -0.052    -0.114     0.009  -0.183 -0.283 -0.080
   MF warm_noisy     -0.005    -0.072     0.056   0.135  0.021  0.244
Saved H7_mode_gap.pdf


In [10]:
# ── Сводная таблица для текста ВКР ──────────────────────────────────
view = (summary
        .pivot_table(index=['model_type', 'init_type'], columns='mode',
                     values='trace_T_mean')
        .reindex(columns=MODES))
print('tr(Σ̂)_T (среднее по seed):')
print(view.round(2).to_string())
print()

view_kl = (summary
           .pivot_table(index=['model_type', 'init_type'], columns='mode',
                        values='kl_T_mean')
           .reindex(columns=MODES))
print('KL(P_T‖P_0) (среднее по seed):')
print(view_kl.round(2).to_string())

tr(Σ̂)_T (среднее по seed):
mode                   closed_loop  static  fresh_oracle  no_influence
model_type init_type                                                  
MF         cold               1.55    1.51          1.55          8.93
           warm               1.50    1.56          1.55          8.15
           warm_noisy         1.54    1.55          1.44          8.19
MLP        cold               0.91    0.77          0.69          6.86
           warm               1.24    1.40          1.21          8.54
           warm_noisy         1.16    1.40          1.10          8.41

KL(P_T‖P_0) (среднее по seed):
mode                   closed_loop  static  fresh_oracle  no_influence
model_type init_type                                                  
MF         cold              15.88   15.98         16.08          7.66
           warm              15.94   16.12         16.09          7.80
           warm_noisy        16.13   16.00         15.93          7.74
MLP        cold  

## Интерпретация

Результаты позволяют ответить на главный вопрос диагностики §19: возникает
ли различие между режимами при «содержательной» инициализации
рекомендателя?

1. **MLP vs MF.** Bilinear MF с identity-init задаёт начальный скор
   $\hat r(u, v) \approx \sigma(\langle u, v\rangle)$ — top-K по такому
   скорy уже на $t=0$ различен для пользователей из разных GMM-кластеров.
   Если разрыв `closed_loop − static` по $\mathrm{tr}\hat\Sigma_T$ и $KL_T$
   расширяется при переходе MLP → MF (см. таблицу `gap_df` и
   `H7_mode_gap.pdf`), это подтверждает H7.

2. **Cold vs Warm.** Warm-start на `true_pref` устраняет «случайный
   старт» модели даже для MLP. Если в режимах `warm`/`warm_noisy` разрыв
   увеличивается по сравнению с `cold` (внутри того же класса модели),
   это второй независимый аргумент за H7.

3. **Bootstrap-CI** в `H7_mode_gap.csv` сообщает, является ли разрыв
   статистически отличимым от нуля по 30 запускам.

Ожидаемая интерпретация в тексте работы:

- если разрыв расширяется — соответствующие графики и таблица идут в
  главу 3 как «контрольный эксперимент к §19»: коллапс `tr(Σ)` *всё ещё*
  объясняется β-дрейфом (это инвариантно), но KL-метрика начинает
  выделять `closed_loop` именно тогда, когда у системы есть структура
  для самоусиления;
- если разрыв не расширяется — это сильный аргумент в пользу финальной
  переформулировки H3 (см. §7.3 в `experiments/diagnostic/REPORT.md`):
  даже информированный рекомендатель не порождает специфического
  «эффекта петли» в имеющейся формализации, и истинный feedback loop
  должен изучаться через `drift_alpha` или альтернативную постановку.
